# Model development

In [ ]:
import pandas as pd
import numpy as np
from eda_functions import *
from sanity_functions import *
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import ADASYN
from sklearn.metrics import precision_recall_curve

In [ ]:
transactions = pd.read_csv('raw_transactions.csv')
transactions.head()

In [ ]:
transactions.shape
# (991.965, 91)

## Null

In [ ]:
import pandas as pd
from scipy import stats

def analyze_nulls_relationship_with_fraud(df):
    """
    Analyze if null values in each column are related to fraud
    by comparing fraud rates and using chi-square test
    """
    results = []

    # Get total fraud rate for reference
    total_fraud_rate = df['infraction'].mean()

    for column in df.columns:
        if column != 'infraction' and df[column].isnull().any():
            # Calculate fraud rates
            fraud_rate_null = df[df[column].isnull()]['infraction'].mean()
            fraud_rate_not_null = df[df[column].notnull()]['infraction'].mean()

            # Create contingency table for chi-square test
            contingency = pd.crosstab(df[column].isnull(), df['infraction'])

            # Perform chi-square test
            chi2, p_value = stats.chi2_contingency(contingency)[:2]

            # Calculate percentage of nulls
            null_percentage = (df[column].isnull().sum() / len(df)) * 100

            # Calculate relative difference in fraud rates
            if fraud_rate_not_null > 0:
                relative_difference = (fraud_rate_null - fraud_rate_not_null) / fraud_rate_not_null
            else:
                relative_difference = np.inf if fraud_rate_null > 0 else 0

            results.append({
                'Column': column,
                'Null_Percentage': null_percentage,
                'Fraud_Rate_Null': fraud_rate_null,
                'Fraud_Rate_Not_Null': fraud_rate_not_null,
                'Relative_Difference': relative_difference,
                'Chi_Square': chi2,
                'P_Value': p_value,
                'Is_Significant': p_value < 0.05,
                'Conclusion': 'Related to fraud' if p_value < 0.05 else 'Random'
            })

    # Convert to DataFrame and sort by significance
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('P_Value')

    # Format percentages and float values
    results_df['Null_Percentage'] = results_df['Null_Percentage'].round(2)
    results_df['Fraud_Rate_Null'] = results_df['Fraud_Rate_Null'].round(4)
    results_df['Fraud_Rate_Not_Null'] = results_df['Fraud_Rate_Not_Null'].round(4)
    results_df['Relative_Difference'] = results_df['Relative_Difference'].round(2)
    results_df['P_Value'] = results_df['P_Value'].apply(lambda x: f'{x:.2e}')

    return results_df

# Load the data
df = pd.read_csv('raw_transactions.csv')

# Run the analysis
results = analyze_nulls_relationship_with_fraud(df)

# Print results
print("\nNull Values Analysis Results:")
print("\nColumns with significant relationship to fraud (p < 0.05):")
significant = results[results['Is_Significant']]
print(significant[['Column', 'Null_Percentage', 'Fraud_Rate_Null', 'Fraud_Rate_Not_Null', 'Relative_Difference', 'P_Value']])

print("\nColumns with random null distribution (p >= 0.05):")
not_significant = results[~results['Is_Significant']]
print(not_significant[['Column', 'Null_Percentage', 'Fraud_Rate_Null', 'Fraud_Rate_Not_Null', 'Relative_Difference', 'P_Value']])

# Save detailed results
results.to_csv('null_analysis_results.csv', index=False)

In [ ]:
caract_df(transactions)

In [ ]:
from sklearn.impute import KNNImputer

def treat_missing_values(df):
    """
    Comprehensive missing value treatment preserving the signal from null values
    """
    # Create a copy to avoid modifying the original dataframe
    df_treated = df.copy()

    # Store columns to process (excluding target and non-numeric columns)
    columns_to_process = df.select_dtypes(include=['float64', 'int64']).columns
    columns_to_process = [col for col in columns_to_process if not col in ['infraction', 'merchant_id', 'event_created_at']]

    # # Create null indicator features
    # for column in columns_to_process:
    #     if df[column].isnull().any():
    #         # Create missing indicator
    #         df_treated[f'{column}_is_null'] = df[column].isnull().astype(int)

    # Different imputation strategies

    # 1. KNN Imputation for features with spatial/behavioral meaning
    knn_imputer = KNNImputer(n_neighbors=5)

    # 2. Forward fill for time-dependent features
    df_treated['event_created_at'] = pd.to_datetime(df_treated['event_created_at'])
    df_treated = df_treated.sort_values('event_created_at')

    for column in columns_to_process:
        if df[column].isnull().any():
            # Calculate the fraud rate difference for null vs non-null
            fraud_rate_null = df[df[column].isnull()]['infraction'].mean()
            fraud_rate_non_null = df[df[column].notnull()]['infraction'].mean()
            ratio = fraud_rate_null / fraud_rate_non_null if fraud_rate_non_null > 0 else np.inf

            # Choose imputation strategy based on fraud rate ratio
            if ratio > 10:  # If nulls are much more likely to be fraud
                # Use a special value (e.g., -999) to preserve the signal
                df_treated[column] = df_treated[column].fillna(-999999)
            else:
                # For time-series like features
                if 'time' in column.lower() or 'date' in column.lower():
                    df_treated[column] = df_treated.groupby('merchant_id')[column].fillna(method='ffill')
                    df_treated[column] = df_treated.groupby('merchant_id')[column].fillna(method='bfill')

                # For amount-related features
                elif 'amount' in column.lower():
                    df_treated[column] = df_treated.groupby('merchant_id')[column].fillna(
                        df_treated.groupby('merchant_id')[column].transform('median')
                    )

                # For other features, use KNN imputation
                else:
                    df_treated[column] = pd.DataFrame(
                        knn_imputer.fit_transform(df_treated[[column]]),
                        columns=[column],
                        index=df_treated.index
                    )

    # Fill any remaining nulls with median
    for column in columns_to_process:
        if df_treated[column].isnull().any():
            df_treated[column] = df_treated[column].fillna(df_treated[column].median())

    return df_treated

# Example usage:
df = pd.read_csv('raw_transactions.csv')

# Get original null counts
null_counts_before = df.isnull().sum()
print("\nNull counts before treatment:")
print(null_counts_before[null_counts_before > 0])

# Apply treatment
df_treated = treat_missing_values(df)

# Get null counts after treatment
null_counts_after = df_treated.isnull().sum()
print("\nNull counts after treatment:")
print(null_counts_after[null_counts_after > 0])

# Print new features created
new_features = [col for col in df_treated.columns if col not in df.columns]
print("\nNew indicator features created:")
print(new_features)

In [ ]:
df_treated

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb

# Resampling methods from imbalanced-learn
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import TomekLinks
from imblearn.combine import SMOTETomek, SMOTEENN

def evaluate_model(y_true, y_pred, y_proba):
    """Calculate various metrics including Precision@100"""
    # Use zero_division=0 to avoid NaNs if no positives are predicted
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # Calculate Precision@100 (top 100 predictions)
    n_samples = len(y_true)
    n_to_select = min(100, n_samples)  # In case we have fewer than 100 samples

    # Use np.argpartition to get top indices based on probability scores
    top_indices = np.argpartition(y_proba, -n_to_select)[-n_to_select:]
    # Use positional indexing (.iloc) to get the corresponding true labels
    precision_at_100 = np.mean(y_true.iloc[top_indices])

    return {
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Precision@100': precision_at_100
    }

def train_and_evaluate(X_train, X_test, y_train, y_test, sampler_name, sampler=None):
    """Train model with given sampler and evaluate performance"""
    print(f"\nProcessing {sampler_name}...")
    try:
        # Apply sampling if sampler is provided (only on the training data)
        if sampler is not None:
            print(f"Applying {sampler_name}...")
            X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
            print(f"Resampled shape: {X_resampled.shape}, Original shape: {X_train.shape}")
        else:
            X_resampled, y_resampled = X_train, y_train

        # Train a LightGBM classifier
        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.01,
            num_leaves=32,
            feature_fraction=0.7,
            bagging_fraction=0.7,
            bagging_freq=5,
            verbose=-1,
            random_state=42
        )

        print("Training model...")
        model.fit(X_resampled, y_resampled)

        # Make predictions on the test set (which remains unaltered)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        # Calculate metrics
        metrics = evaluate_model(y_test, y_pred, y_proba)

        # Print results
        print(f"\nResults for {sampler_name}:")
        for metric, value in metrics.items():
            print(f"{metric}: {value:.4f}")

        return metrics

    except Exception as e:
        print(f"Error processing {sampler_name}: {str(e)}")
        return {
            'Precision': np.nan,
            'Recall': np.nan,
            'F1-Score': np.nan,
            'Precision@100': np.nan
        }

def main():
    # Load and preprocess data
    print("Loading data...")
    # df = pd.read_csv('raw_transactions.csv')
    df = df_treated.copy()

    # Prepare features (dropping columns that are not used for prediction)
    X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
    y = df['infraction']

    print(f"Data shape: {X.shape}")
    print(f"Number of fraud cases: {sum(y == 1)}")
    print(f"Fraud ratio: {sum(y == 1)/len(y):.4%}")

    # Train-test split (stratify by y to preserve class distribution)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Training set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Convert back to DataFrame (to preserve column names and allow .iloc indexing)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

    # Define samplers to test with a target ratio of 20% fraud in the training set.
    # For samplers that support sampling_strategy, set sampling_strategy=0.2.
    samplers = {
        'No Sampling': None,
        'Random Under-Sampling': RandomUnderSampler(sampling_strategy=0.2, random_state=42),
        'SMOTE': SMOTE(sampling_strategy=0.2, random_state=42),
        'ADASYN': ADASYN(sampling_strategy=0.2, random_state=42),
        'Tomek Links': TomekLinks(),  # Tomek Links acts as a cleaning step
        'SMOTE-Tomek': SMOTETomek(sampling_strategy=0.2, random_state=42),
        'SMOTE-ENN': SMOTEENN(sampling_strategy=0.2, random_state=42)
    }

    # Store results for comparison
    results = {}

    # Test each sampling method
    for name, sampler in samplers.items():
        metrics = train_and_evaluate(
            X_train_scaled, X_test_scaled,
            y_train, y_test,
            name, sampler
        )
        results[name] = metrics

    # Create a comparison DataFrame
    comparison_df = pd.DataFrame(results).T

    # Save the comparison results
    comparison_df.to_csv('sampling_comparison.csv', index=True)
    print("\nFull comparison:")
    print(comparison_df)

    return comparison_df

if __name__ == "__main__":
    comparison_df = main()


In [ ]:
# MODIFIED CODE WITH KEY FIXES
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
import lightgbm as lgb
from sklearn.impute import SimpleImputer


def evaluate_model(y_true, y_pred, y_proba):
    metrics = {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'Avg Precision': average_precision_score(y_true, y_proba),  # New metric
    }

    # Precision@100 using pandas for index alignment
    y_proba_series = pd.Series(y_proba, index=y_true.index)
    top_100 = y_proba_series.nlargest(100)
    metrics['Precision@100'] = y_true.loc[top_100.index].mean()

    return metrics

# Modified training function with proper data handling
def train_and_evaluate(X_train, X_test, y_train, y_test, sampler_name, sampler=None):
    print(f"\nProcessing {sampler_name}...")

    try:
        # 1. Impute missing values first
        imputer = SimpleImputer(strategy='median')
        X_train_imputed = imputer.fit_transform(X_train)
        X_test_imputed = imputer.transform(X_test)

        # 2. Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_imputed)
        X_test_scaled = scaler.transform(X_test_imputed)

        # 3. Apply sampling only on training data
        if sampler is not None:
            # Preserve dataframe structure for feature names
            X_res, y_res = sampler.fit_resample(
                pd.DataFrame(X_train_scaled, columns=X_train.columns),
                y_train
            )
            print(f"Class balance after resampling: {y_res.mean():.2%}")
        else:
            X_res, y_res = X_train_scaled, y_train

        # 4. Train LightGBM with balanced class weights
        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.01,
            num_leaves=32,
            scale_pos_weight=len(y_res[y_res==0])/len(y_res[y_res==1]),
            random_state=42,
            verbose=-1
        )

        model.fit(X_res, y_res)

        # 5. Predict on original test set
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]

        return evaluate_model(y_test, y_pred, y_proba)

    except Exception as e:
        print(f"Error: {str(e)}")
        return {k: np.nan for k in ['Precision', 'Recall', 'F1-Score', 'Precision@100']}

# Modified samplers with explicit 20% fraud ratio target
samplers = {
    'No Sampling': None,
    'Random Under-Sampling': RandomUnderSampler(
        sampling_strategy=0.25,  # Minority = 25% of majority (20% total)
        random_state=42
    ),
    'SMOTE': SMOTE(
        sampling_strategy=0.25,
        random_state=42
    ),
    'ADASYN': ADASYN(
        sampling_strategy=0.25,
        random_state=42
    ),
    'SMOTE-Tomek': SMOTETomek(
        sampling_strategy=0.25,
        random_state=42
    ),
    'SMOTE-ENN': SMOTEENN(
        sampling_strategy=0.25,
        random_state=42
    )
}

# Execute the modified pipeline
def main():
    # Load data with proper datetime handling
    # df = pd.read_csv('raw_transactions.csv').sort_values('event_created_at')
    df = df_treated.copy().sort_values('event_created_at')

    # Feature engineering
    X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
    y = df['infraction']

    # Time-based split (no shuffle)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    results = {}
    for name, sampler in samplers.items():
        results[name] = train_and_evaluate(X_train, X_test, y_train, y_test, name, sampler)

    results_df = pd.DataFrame(results).T
    print(results_df)
    return results_df

if __name__ == "__main__":
    results = main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
import lightgbm as lgb

def evaluate_model(y_true, y_pred, y_proba):
    """Calculate evaluation metrics including Average Precision"""
    metrics = {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'Avg Precision': average_precision_score(y_true, y_proba),
    }

    # Calculate Precision@100 using pandas for index alignment
    y_proba_series = pd.Series(y_proba, index=y_true.index)
    top_100 = y_proba_series.nlargest(100)
    metrics['Precision@100'] = y_true.loc[top_100.index].mean()

    return metrics

def train_and_evaluate(X_train, X_test, y_train, y_test, sampler_name, sampler=None):
    """Train and evaluate model with given sampler"""
    print(f"\nProcessing {sampler_name}...")

    try:
        # Apply sampling only on training data
        if sampler is not None:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            print(f"Resampled shape: {X_res.shape}, Fraud ratio: {y_res.mean():.4f}")
        else:
            X_res, y_res = X_train, y_train

        # Train LightGBM with dynamic class weights
        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.01,
            num_leaves=32,
            scale_pos_weight=len(y_res[y_res==0])/len(y_res[y_res==1]),
            random_state=42,
            verbose=-1
        )

        model.fit(X_res, y_res)

        # Generate predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        return evaluate_model(y_test, y_pred, y_proba)

    except Exception as e:
        print(f"Error: {str(e)}")
        return {k: np.nan for k in ['Precision', 'Recall', 'F1-Score',
                                   'Avg Precision', 'Precision@100']}

def main():
    """Main execution flow"""
    # Load and prepare data
    # df = pd.read_parquet('sumup_cas.parquet').sort_values('event_created_at')
    df = df_treated.sort_values('event_created_at').copy()

    # Feature selection
    X = df.drop(['infraction', 'event_created_at', 'merchant_id'], axis=1)
    y = df['infraction']

    # Time-based split (no shuffle)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    # Define samplers with 20% fraud target
    samplers = {
        'No Sampling': None,
        'Random Under': RandomUnderSampler(sampling_strategy=0.25, random_state=42),
        'SMOTE': SMOTE(sampling_strategy=0.25, random_state=42),
        'ADASYN': ADASYN(sampling_strategy=0.25, random_state=42),
        'SMOTE-Tomek': SMOTETomek(sampling_strategy=0.25, random_state=42),
        'SMOTE-ENN': SMOTEENN(sampling_strategy=0.25, random_state=42)
    }

    # Evaluate all samplers
    results = {}
    for name, sampler in samplers.items():
        results[name] = train_and_evaluate(X_train, X_test, y_train, y_test, name, sampler)

    # Save and display results
    results_df = pd.DataFrame(results).T
    results_df.to_csv('sampling_results2.csv', float_format='%.4f')

    print("\nFinal Results:")
    print(results_df)

    return results_df

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
import lightgbm as lgb
import xgboost as xgb

def evaluate_model(model_name, y_true, y_pred, y_proba):
    """Calculate evaluation metrics with model name prefix"""
    metrics = {
        f'{model_name}_Precision': precision_score(y_true, y_pred, zero_division=0),
        f'{model_name}_Recall': recall_score(y_true, y_pred, zero_division=0),
        f'{model_name}_F1-Score': f1_score(y_true, y_pred, zero_division=0),
        f'{model_name}_Avg Precision': average_precision_score(y_true, y_proba),
    }

    # Calculate Precision@100
    y_proba_series = pd.Series(y_proba, index=y_true.index)
    top_100 = y_proba_series.nlargest(100)
    metrics[f'{model_name}_Precision@100'] = y_true.loc[top_100.index].mean()

    return metrics

def train_and_evaluate(X_train, X_test, y_train, y_test, sampler_name, sampler=None):
    """Train and evaluate both models with given sampler"""
    print(f"\nProcessing {sampler_name}...")
    results = {}

    try:
        # Apply sampling only on training data
        if sampler is not None:
            X_res, y_res = sampler.fit_resample(X_train, y_train)
            print(f"Resampled shape: {X_res.shape}, Fraud ratio: {y_res.mean():.4f}")
        else:
            X_res, y_res = X_train, y_train

        class_ratio = len(y_res[y_res==0])/len(y_res[y_res==1])

        # ========== LightGBM ==========
        try:
            lgb_model = lgb.LGBMClassifier(
                n_estimators=1000,
                learning_rate=0.01,
                num_leaves=32,
                scale_pos_weight=class_ratio,
                random_state=42,
                verbose=-1
            )
            lgb_model.fit(X_res, y_res)
            lgb_pred = lgb_model.predict(X_test)
            lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
            results.update(evaluate_model('LGBM', y_test, lgb_pred, lgb_proba))
        except Exception as e:
            print(f"LightGBM Error: {str(e)}")

        # ========== XGBoost ==========
        try:
            xgb_model = xgb.XGBClassifier(
                n_estimators=1000,
                learning_rate=0.01,
                max_depth=5,
                scale_pos_weight=class_ratio,
                eval_metric='aucpr',
                random_state=42,
                use_label_encoder=False
            )
            xgb_model.fit(X_res, y_res)
            xgb_pred = xgb_model.predict(X_test)
            xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
            results.update(evaluate_model('XGB', y_test, xgb_pred, xgb_proba))
        except Exception as e:
            print(f"XGBoost Error: {str(e)}")

        return results

    except Exception as e:
        print(f"General Error: {str(e)}")
        return {k: np.nan for k in [
            'LGBM_Precision', 'LGBM_Recall', 'LGBM_F1-Score', 'LGBM_Avg Precision', 'LGBM_Precision@100',
            'XGB_Precision', 'XGB_Recall', 'XGB_F1-Score', 'XGB_Avg Precision', 'XGB_Precision@100'
        ]}

# Rest of the main() function remains the same...

# Example output format:
"""
                   LGBM_Precision  LGBM_Recall  ...  XGB_Avg Precision  XGB_Precision@100
No Sampling                0.0012       0.8000  ...             0.0124             0.1500
Random Under               0.1200       0.7500  ...             0.2105             0.3200
...                         ...          ...  ...                ...                ...
"""